# SVC — 조합 Pilot

공개된 4개 조합 조건을 재현하기 위한 코드. 기존 실측 결과와 별개이며 아직 신규 재실행하지 않음.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("SVC_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("SVC", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
sbase=base
print("Local output:",root)

In [ ]:
t=time.perf_counter()
Xs,ys,meta=fb.train_features('SVC_Combined',off,wr)
print(Xs.shape,meta,time.perf_counter()-t)
np.save(root/'runtime/svc_X.npy',Xs)
np.save(root/'runtime/svc_y.npy',ys)
np.save(root/'runtime/dev_X.npy',Xv)
np.save(root/'runtime/dev_y.npy',y)
np.save(root/'runtime/official_X.npy',Xo)
np.save(root/'runtime/official_y.npy',yo)

In [ ]:
# Predeclare pilot for resource-limited SVC screen (explicit not full DOE). 
# Save exact training selections and augmentation candidate (no validation used in generation).
rng=np.random.default_rng(42)
# sample balanced 500 per class official, 4 per class/cohort per origin/aug, 2 per class/cohort for targeted
pilot=[]
for c0 in range(16):
 pilot.extend(rng.choice(np.flatnonzero(ys[:76800]==c0),500,replace=False))
for offset in [76800,79360]:
 for w0 in range(4):
  for c0 in range(16):
   pool=np.flatnonzero(yh[w0*640:(w0+1)*640]==c0)+w0*640
   pilot.extend(offset+rng.choice(pool,4,replace=False))
# exact target source indices used by baseline
rngt=np.random.default_rng(fb.TARGET_SEED)
target_groups=[]
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)==w)&(yh==c0))
  target_groups.append(rngt.permutation(pool)[:20])
ti=np.sort(np.concatenate(target_groups))
assert np.array_equal(ys[81920:],yh[ti])
# four-cohort target sampled 2 per class/cohort
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)[ti]==w)&(yh[ti]==c0))
  pilot.extend(81920+rng.choice(pool,2,replace=False))
pilot=np.sort(np.array(pilot))
assert len(pilot)==8528
# predeclared pilot official eval balanced 40/class =640 (not full)
oid=np.sort(np.concatenate([rng.choice(np.flatnonzero(yo==c0),40,replace=False) for c0 in range(16)]))
np.savez(root/'02_SVC/pilot_indices.npz',training_rows=pilot,official_validation_rows=oid)
proto['SVC']['pilot_resource_adaptation']={'rows':len(pilot),'official_train':8000,'handmade_original':256,'handmade_augmented':256,'handmade_targeted':16,'official_validation':640,'handmade_validation':1280,'purpose':'screening only; full 9x DOE not claimed complete; full fit required for adoption'}
(root/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(proto,ensure_ascii=False,indent=2))
print(len(pilot),len(oid),np.bincount(ys[pilot]))
# set up data 
Xp=Xs[pilot];yp=ys[pilot];hp=(pilot>=76800)
gscale=1/(Xp.shape[1]*Xp.var())
print('pilot gamma',gscale)

In [ ]:
from sklearn.svm import SVC
spdir=root/'02_SVC'
sprows=[]
sppreds={}
def svc_pilot(name,exp,C=1.,gm=1.,hand_w=1.,target_w=1.,xx=Xp,yy=yp,handmask=hp):
    t=time.perf_counter()
    gamma=float(gm/(xx.shape[1]*xx.var()))
    weights=np.ones(len(yy),np.float64); weights[handmask]*=hand_w
    weights[handmask & np.isin(yy,[labels.index(c) for c in 'BDC'])]*=target_w
    m=SVC(C=C,gamma=gamma,cache_size=200)
    m.fit(xx,yy,sample_weight=weights)
    fit=time.perf_counter()-t
    t=time.perf_counter(); pv=m.predict(Xv); vo=m.predict(Xo[oid]); infer=time.perf_counter()-t
    res={'condition':name,'experiment':exp,'scope':'pilot','train_rows':len(yy),'C':C,'gamma_multiplier':gm,'gamma':gamma,'hand_weight':hand_w,'target_weight':target_w,'fit_s':fit,'batch_eval_s':infer,'official_pilot_n':len(oid),'official_pilot_correct':int((vo==yo[oid]).sum()),**stats(pv,sbase)}
    sprows.append(res);sppreds[name]=pv
    pd.DataFrame(sprows).to_csv(spdir/'pilot_all_conditions.csv',index=False)
    pd.DataFrame({'true':y,**sppreds}).to_csv(spdir/'pilot_predictions.csv',index=False)
    print(name,res['correct'],round(res['macro_f1'],5),res['official_pilot_correct'],round(fit,2),flush=True)
    return res

조합 실행 셀은 공개된 조건 CSV와 원본 svc_pilot 함수를 이용해 새로 구성한 재현 코드이며 원래 별도 실행 셀의 바이트 복사본이 아니다. Full 조합 학습은 1차 기록에서 미실시.

In [ ]:
svc_pilot("C1_G1","004")
svc_pilot("C2_G2","004",C=2,gm=2)
for name,C,gm,hw,tw in [
    ("C2G2_TW2",2,2,1.,2.),("C2G2_SW1.5",2,2,1.5,1.),
    ("SW1.5_TW2",1,1,1.5,2.),("C2G2_SW1.5_TW2",2,2,1.5,2.)]:
    svc_pilot(name,"004",C=C,gm=gm,hand_w=hw,target_w=tw)
df=pd.DataFrame(sprows);control=df.iloc[0];best=df.iloc[1]
df["promote_full"]=(df.correct>best.correct)&(df.macro_f1>best.macro_f1)&(df.macro_precision>=control.macro_precision)&(df.official_pilot_correct>=621)
df.iloc[2:].to_csv(spdir/"EXP-FT-SVC-004_combination_pilot.csv",index=False)
print(df.iloc[2:][["condition","correct","macro_f1","promote_full"]])